# 지도학습 회귀 — 주문별 판매금액 예측 재현 실험

미완료 과제였던 이상치 유지/제거 비교를 동일한 시간 분할로 실행한다. 비선형 트리 모델에서는 단순 상관계수만으로 변수를 제거하지 않는다.

- 실행 기준 시각: `20260814_062642`
- 원본 노트북은 `01_원본보관`에 수정 없이 보관했다.
- 이 노트북은 최종 재현 파이프라인이다. 과거의 모든 탐색 실험과 출력은 원본 보관본에서 확인할 수 있다.
- 모델 선택은 검증 세트에서만 수행하고, 테스트 세트는 최종 보고에 사용한다.
- 모든 비교는 동일 분할과 동일 평가지표를 사용하며 학습·추론 시간도 기록한다.

## 1. 환경·데이터 로드 및 시간 순서 분할

In [1]:
from pathlib import Path
from time import perf_counter
import json, warnings
import numpy as np
import pandas as pd
import kagglehub
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd()
ARTIFACTS = ROOT / "03_실험결과"
ARTIFACTS.mkdir(exist_ok=True)

data_path = Path(kagglehub.dataset_download("likithagedipudi/starbucks-customer-ordering-patterns"))
csv_path = data_path / "starbucks_customer_ordering_patterns.csv"
df = pd.read_csv(csv_path)
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values(["order_date", "order_time", "order_id"]).reset_index(drop=True)

n = len(df)
i1, i2 = int(n * 0.70), int(n * 0.85)
train_raw = df.iloc[:i1].copy()
val_raw = df.iloc[i1:i2].copy()
test_raw = df.iloc[i2:].copy()
print("data:", csv_path.name, "shape:", df.shape)
print({"train": train_raw.shape, "validation": val_raw.shape, "test": test_raw.shape})
print({"train_end": str(train_raw.order_date.max().date()),
       "validation_end": str(val_raw.order_date.max().date()),
       "test_end": str(test_raw.order_date.max().date())})

data: starbucks_customer_ordering_patterns.csv shape: (100000, 20)
{'train': (70000, 20), 'validation': (15000, 20), 'test': (15000, 20)}
{'train_end': '2025-05-26', 'validation_end': '2025-09-12', 'test_end': '2025-12-30'}


## 2. 누수 방지 전처리

- 무작위 행 분할 대신 날짜순 분할로 미래 주문을 과거 주문으로 예측한다.
- 고객별 주문 수·평균 만족도는 훈련 기간에서만 계산하고 검증/테스트에 매핑한다.
- 기존 노트북에서 파생변수 merge가 반복되어 생긴 `_x`, `_y` 중복 열을 만들지 않는다.
- 식별자만 제거하고, 낮은 선형 상관계수만을 이유로 변수는 삭제하지 않는다.

In [2]:
target = "total_spend"
cat_cols = ["day_of_week", "order_channel", "store_location_type", "region",
            "customer_age_group", "customer_gender", "drink_category"]

customer_stats = train_raw.groupby("customer_id").agg(
    historical_total_orders=("order_id", "count"),
    historical_avg_satisfaction=("customer_satisfaction", "mean"),
)
default_orders = float(customer_stats["historical_total_orders"].median())
default_satisfaction = float(customer_stats["historical_avg_satisfaction"].median())

def build_features(frame):
    z = frame.copy()
    z["historical_total_orders"] = z["customer_id"].map(customer_stats["historical_total_orders"]).fillna(default_orders)
    z["historical_avg_satisfaction"] = z["customer_id"].map(customer_stats["historical_avg_satisfaction"]).fillna(default_satisfaction)
    z["order_year"] = z["order_date"].dt.year
    z["order_month"] = z["order_date"].dt.month
    z["order_day"] = z["order_date"].dt.day
    times = pd.to_datetime(z["order_time"], format="%H:%M")
    z["order_hour"] = times.dt.hour
    z["order_minute"] = times.dt.minute
    y = z[target].astype(float)
    z = z.drop(columns=[target, "customer_id", "order_id", "order_date", "order_time", "store_id"])
    for col in z.select_dtypes(include="bool").columns:
        z[col] = z[col].astype(int)
    z = pd.get_dummies(z, columns=cat_cols, drop_first=False, dtype=int)
    return z, y

X_train, y_train = build_features(train_raw)
X_val, y_val = build_features(val_raw)
X_test, y_test = build_features(test_raw)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

assert not any(c.endswith("_x") or c.endswith("_y") for c in X_train.columns)
assert target not in X_train.columns
print("feature_count:", X_train.shape[1], "duplicate_feature_names:", int(X_train.columns.duplicated().sum()))

feature_count: 48 duplicate_feature_names: 0


## 3. 미완료 실험: 이상치 유지 vs 훈련 데이터에서만 제거

In [3]:
q1, q3 = y_train.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
keep_mask = y_train.between(lower, upper)
print({"lower": lower, "upper": upper, "train_outliers": int((~keep_mask).sum()),
       "train_rows_keep": len(y_train), "train_rows_removed_variant": int(keep_mask.sum())})

xgb_params = dict(
    n_estimators=300, learning_rate=0.03, max_depth=5, subsample=0.8,
    colsample_bytree=0.9, gamma=0.1, reg_alpha=0.005,
    objective="reg:squarederror", random_state=SEED, n_jobs=-1,
)
lgbm_params = dict(
    n_estimators=350, learning_rate=0.04, num_leaves=31,
    subsample=0.85, colsample_bytree=0.9,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)

models = {
    "Linear_keep": (LinearRegression(), X_train, y_train),
    "XGB_keep": (XGBRegressor(**xgb_params), X_train, y_train),
    "XGB_remove_train_outliers": (XGBRegressor(**xgb_params), X_train.loc[keep_mask], y_train.loc[keep_mask]),
    "LGBM_keep": (LGBMRegressor(**lgbm_params), X_train, y_train),
    "LGBM_remove_train_outliers": (LGBMRegressor(**lgbm_params), X_train.loc[keep_mask], y_train.loc[keep_mask]),
}

fit_seconds = {}
val_pred, test_pred = {}, {}
infer_seconds = {}
for name, (model, xf, yf) in models.items():
    start = perf_counter(); model.fit(xf, yf); fit_seconds[name] = perf_counter() - start
    val_pred[name] = model.predict(X_val)
    start = perf_counter(); test_pred[name] = model.predict(X_test); infer_seconds[name] = perf_counter() - start
    print(name, "fit_seconds=", round(fit_seconds[name], 3))

val_pred["Blend_XGB_LGBM_keep"] = (val_pred["XGB_keep"] + val_pred["LGBM_keep"]) / 2
test_pred["Blend_XGB_LGBM_keep"] = (test_pred["XGB_keep"] + test_pred["LGBM_keep"]) / 2
fit_seconds["Blend_XGB_LGBM_keep"] = fit_seconds["XGB_keep"] + fit_seconds["LGBM_keep"]
infer_seconds["Blend_XGB_LGBM_keep"] = infer_seconds["XGB_keep"] + infer_seconds["LGBM_keep"]

val_pred["Blend_XGB_LGBM_remove_train_outliers"] = (
    val_pred["XGB_remove_train_outliers"] + val_pred["LGBM_remove_train_outliers"]
) / 2
test_pred["Blend_XGB_LGBM_remove_train_outliers"] = (
    test_pred["XGB_remove_train_outliers"] + test_pred["LGBM_remove_train_outliers"]
) / 2
fit_seconds["Blend_XGB_LGBM_remove_train_outliers"] = (
    fit_seconds["XGB_remove_train_outliers"] + fit_seconds["LGBM_remove_train_outliers"]
)
infer_seconds["Blend_XGB_LGBM_remove_train_outliers"] = (
    infer_seconds["XGB_remove_train_outliers"] + infer_seconds["LGBM_remove_train_outliers"]
)

{'lower': -0.14000000000000057, 'upper': 29.14, 'train_outliers': 1024, 'train_rows_keep': 70000, 'train_rows_removed_variant': 68976}
Linear_keep fit_seconds= 0.084


XGB_keep fit_seconds= 1.372


XGB_remove_train_outliers fit_seconds= 1.139


LGBM_keep fit_seconds= 0.961


LGBM_remove_train_outliers fit_seconds= 1.197


In [4]:
def regression_metrics(y_true, pred):
    mse = mean_squared_error(y_true, pred)
    return {"mae": mean_absolute_error(y_true, pred), "mse": mse,
            "rmse": np.sqrt(mse), "r2": r2_score(y_true, pred)}

rows = []
for name in val_pred:
    vm = regression_metrics(y_val, val_pred[name])
    tm = regression_metrics(y_test, test_pred[name])
    row = {"model": name, "fit_seconds": fit_seconds[name],
           "inference_ms_per_1000": infer_seconds[name] * 1_000_000 / len(X_test)}
    row.update({f"val_{k}": v for k, v in vm.items()})
    row.update({f"test_{k}": v for k, v in tm.items()})
    rows.append(row)

results = pd.DataFrame(rows).sort_values("val_rmse").reset_index(drop=True)
# 최소 검증 RMSE와 0.1% 이내인 후보 중 추론이 가장 빠른 모델을 선택한다.
best_val_rmse = results["val_rmse"].min()
near_best = results[results["val_rmse"] <= best_val_rmse * 1.001]
selected_name = near_best.sort_values("inference_ms_per_1000").iloc[0]["model"]
print("absolute_best_validation_rmse:", results.loc[0, "model"])
print("selected_with_0.1pct_parsimony_rule:", selected_name)
display(results[["model", "val_rmse", "test_mae", "test_rmse", "test_r2",
                 "fit_seconds", "inference_ms_per_1000"]])
results.to_csv(ARTIFACTS / "regression_experiment_results.csv", index=False)

predictions = pd.DataFrame({
    "order_date": test_raw["order_date"].dt.strftime("%Y-%m-%d").values,
    "actual_total_spend": y_test.values,
    "baseline_linear_prediction": test_pred["Linear_keep"],
    "selected_prediction": test_pred[selected_name],
})
predictions.to_csv(ARTIFACTS / "regression_test_predictions.csv", index=False)

absolute_best_validation_rmse: Blend_XGB_LGBM_keep
selected_with_0.1pct_parsimony_rule: XGB_keep


,model,val_rmse,test_mae,test_rmse,test_r2,fit_seconds,inference_ms_per_1000
0,Blend_XGB_LGBM_keep,0.921624,0.801332,0.928736,0.971490,2.332687,4.651573
1,XGB_keep,0.921724,0.800736,0.927956,0.971538,1.372078,1.012133
2,LGBM_keep,0.922651,0.802612,0.930597,0.971376,0.960609,3.639440
3,Linear_keep,0.976771,0.831626,0.978436,0.968357,0.084342,0.255007
4,XGB_remove_train_outliers,0.999272,0.826207,0.997525,0.967110,1.138811,1.097193
5,Blend_XGB_LGBM_remove_train_outliers,1.003290,0.827457,1.000983,0.966882,2.335532,3.270000
6,LGBM_remove_train_outliers,1.008767,0.829387,1.005931,0.966554,1.196721,2.172807


## 4. 개선폭의 불확실성: paired bootstrap 95% 신뢰구간

In [5]:
rng = np.random.default_rng(SEED)
y_arr = y_test.to_numpy()
base = test_pred["Linear_keep"]
selected = test_pred[selected_name]
rmse_reductions = []
for _ in range(1000):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    base_rmse = np.sqrt(mean_squared_error(y_arr[idx], base[idx]))
    selected_rmse = np.sqrt(mean_squared_error(y_arr[idx], selected[idx]))
    rmse_reductions.append(base_rmse - selected_rmse)
ci = np.quantile(rmse_reductions, [0.025, 0.975])
summary = {
    "selected_model": selected_name,
    "selection_rule": "within 0.1% of minimum validation RMSE, then minimum inference time",
    "test_rmse_reduction_vs_linear": float(np.mean(rmse_reductions)),
    "bootstrap_95pct_ci": [float(ci[0]), float(ci[1])],
    "time_split": True,
    "test_rows": int(len(y_test)),
    "training_outlier_count": int((~keep_mask).sum()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(ARTIFACTS / "regression_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "selected_model": "XGB_keep",
  "selection_rule": "within 0.1% of minimum validation RMSE, then minimum inference time",
  "test_rmse_reduction_vs_linear": 0.05049356396282084,
  "bootstrap_95pct_ci": [
    0.045770204307081674,
    0.055643206175501635
  ],
  "time_split": true,
  "test_rows": 15000,
  "training_outlier_count": 1024
}


341

## 5. 의사결정 원칙

- 이상치는 오류라고 단정할 근거가 없으므로 테스트 데이터에서는 제거하지 않는다.
- `유지`와 `훈련에서만 제거`의 검증 RMSE를 같은 분할에서 비교하고 더 나은 쪽을 선택한다.
- 앙상블이 단일 모델보다 근소하게 나아도 학습·추론 비용을 함께 제시한다.
- 이 데이터셋은 합성 데이터이므로 실제 매출 운영에 적용하기 전 실거래 데이터에서 외부 검증이 필요하다.